In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

In [0]:
bronze_df = spark.table("weather_raw")
bronze_df.limit(5).display()

In [0]:
silver_clean = (
    bronze_df.select(
        F.col("`loc_id`").cast("int").alias("loc_id"),
        F.to_date(F.col("`date`")).alias("date"),
        F.col("`avgtempC`").cast("int").alias("avg_temp_c"),
        F.col("`totalprecipMM`").cast("double").alias("total_precip_mm"),
        F.col("`windspeedKmph`").cast("int").alias("wind_speed_kmph"),
        F.col("`humidity`").cast("int").alias("humidity"),
        F.col("`visibilityKm`").cast("int").alias("visibility_km"),
        F.col("`cloudcover`").cast("int").alias("cloud_cover"),
        F.col("`pressureMB`").cast("int").alias("pressure_mb"),
        F.col("`weatherDesc`").cast("string").alias("weather_desc")
    )
)

silver_clean.limit(5).display()

In [0]:
silver_dedup = silver_clean.dropDuplicates(["date", "loc_id"])

# Data quality expectations
silver_validated = (
    silver_dedup
    .filter((F.col("humidity").between(0, 100)))
    .filter((F.col("cloud_cover").between(0, 100)))
    .filter((F.col("avg_temp_c").isNotNull()) & (F.col("date").isNotNull()))
)

# Save to table
silver_validated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("students_data.team1_taxi.silver_weather")

print(f"silver_weather: {silver_validated.count()} rows")

In [0]:
weather_daily = (
    silver_validated
    .groupBy("date")
    .agg(
        F.avg("avg_temp_c").alias("avg_temp_c"),
        F.sum("total_precip_mm").alias("total_precip_mm"),
        F.avg("wind_speed_kmph").alias("avg_wind_speed_kmph"),
        F.avg("humidity").alias("avg_humidity"),
        F.avg("visibility_km").alias("avg_visibility_km"),
        F.avg("cloud_cover").alias("avg_cloud_cover"),
        F.avg("pressure_mb").alias("avg_pressure_mb"),
        F.first("weather_desc").alias("weather_desc")  # Or mode
    )
    .orderBy("date")
)